# Gerenciamento Inteligente de um Sistema de Produção
## MDP + Value Iteration + Q-Learning com NASA C-MAPSS

**Aluno:** Ialisson Roque da Silva  
**Curso:** Engenharia Eletrônica — UFRPE/UACSA  
**Ano:** 2026

Este notebook implementa o pipeline completo do projeto:

**C-MAPSS → pré-processamento → RUL → normalização por condição operacional → seleção de sensores → PCA / Health Index → estados discretos → transições empíricas → MDP → Value Iteration → Q-Learning → comparação de políticas.**

> **Importante:** o C-MAPSS é usado como fonte de trajetórias simuladas de degradação. Ele não representa fisicamente uma linha de produção. As ações de manutenção, seus efeitos e custos são hipóteses do ambiente de decisão.


## 1. Bibliotecas

Caso seja necessário instalar as dependências, execute:

`pip install numpy pandas scipy scikit-learn matplotlib`


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Ambiente configurado.")

## 2. Configuração dos dados

Coloque na mesma pasta do notebook os arquivos:

- `train_FD001.txt` ... `train_FD004.txt`
- `test_FD001.txt` ... `test_FD004.txt`
- `RUL_FD001.txt` ... `RUL_FD004.txt`

O código também aceita os arquivos de teste com sufixo `(1)`, como `test_FD001(1).txt`.


In [ ]:
DATA_DIR = Path(".")

COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
SENSORS = [f"s{i}" for i in range(1, 22)]

ACTIONS = ["operate", "reduced_load", "maintenance"]
STATE_NAMES = ["healthy", "moderate", "severe", "critical", "failure"]

def load_fd(folder, k):
    folder = Path(folder)
    train = pd.read_csv(
        folder / f"train_FD00{k}.txt",
        sep=r"\s+", header=None, names=COLS
    )

    test_path = folder / f"test_FD00{k}.txt"
    if not test_path.exists():
        test_path = folder / f"test_FD00{k}(1).txt"

    test = pd.read_csv(test_path, sep=r"\s+", header=None, names=COLS)
    rul = np.loadtxt(folder / f"RUL_FD00{k}.txt")
    return train, test, rul

## 3. Inspeção inicial

O C-MAPSS possui **3 configurações operacionais e 21 sensores** por ciclo.

FD001 e FD003 possuem uma condição operacional; FD002 e FD004 possuem múltiplas condições. Nos conjuntos com múltiplas condições, os regimes são identificados antes da normalização dos sensores.


In [ ]:
summary = []
datasets = {}

for k in range(1, 5):
    train, test, rul = load_fd(DATA_DIR, k)
    datasets[k] = {"train": train, "test": test, "rul": rul}
    summary.append({
        "dataset": f"FD00{k}",
        "train_rows": len(train),
        "train_units": train["unit"].nunique(),
        "test_rows": len(test),
        "test_units": test["unit"].nunique(),
        "RUL_values": len(np.atleast_1d(rul)),
        "max_train_cycle": train["cycle"].max()
    })

summary_df = pd.DataFrame(summary)
summary_df

### Observação sobre FD004

Nos arquivos usados no projeto, o FD004 possui **249 unidades de treinamento e 248 de teste**. Isso difere da contagem indicada no README recebido. O projeto preserva os arquivos fornecidos e documenta a divergência em vez de alterá-los.


## 4. RUL, normalização e Health Index

No conjunto de treinamento, o RUL é calculado como:

\[
RUL_{i,t} = T_i - t
\]

onde \(T_i\) é o último ciclo da unidade \(i\).

Para FD002 e FD004, as três configurações operacionais são agrupadas em **6 regimes** com K-Means. Os sensores são então normalizados dentro de cada regime.

Os 10 sensores com maior correlação de Spearman absoluta com o RUL são selecionados. Em seguida, PCA reduz essas variáveis a um componente principal. O componente é orientado e normalizado para formar o **Health Index (HI)**.


In [ ]:
def prepare_train(df, n_conditions):
    d = df.copy()
    d["RUL"] = d.groupby("unit")["cycle"].transform("max") - d["cycle"]

    if n_conditions == 1:
        d["condition"] = 0
    else:
        ops = StandardScaler().fit_transform(d[["op1", "op2", "op3"]])
        d["condition"] = KMeans(
            n_clusters=n_conditions, random_state=42, n_init=10
        ).fit_predict(ops)

    Z = pd.DataFrame(index=d.index)

    for sensor in SENSORS:
        z = np.zeros(len(d))
        for _, idx in d.groupby("condition").groups.items():
            values = d.loc[idx, sensor]
            sd = values.std()
            z[idx] = 0.0 if sd < 1e-8 else (values - values.mean()) / sd
        Z[sensor] = z

    corr = {
        sensor: spearmanr(Z[sensor], d["RUL"]).statistic
        for sensor in SENSORS if Z[sensor].std() > 1e-8
    }

    top_sensors = sorted(
        corr, key=lambda s: abs(corr[s]), reverse=True
    )[:10]

    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(Z[top_sensors]).ravel()

    # Orienta PC1 para aumentar quando o RUL aumenta.
    if spearmanr(pc1, d["RUL"]).statistic < 0:
        pc1 = -pc1

    lo, hi = np.percentile(pc1, [1, 99])
    d["HI_raw"] = np.clip((pc1 - lo) / (hi - lo), 0, 1)

    # Suavização temporal.
    d["HI"] = d.groupby("unit")["HI_raw"].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )

    # 0 saudável, 1 moderado, 2 severo, 3 crítico.
    d["state"] = pd.cut(
        d["HI"],
        [-np.inf, .25, .50, .75, np.inf],
        labels=[3, 2, 1, 0]
    ).astype(int)

    return d, top_sensors, corr, pca.explained_variance_ratio_[0]

## 5. Construção do Health Index nos quatro subconjuntos


In [ ]:
prepared = {}
hi_results = []

for k in range(1, 5):
    train = datasets[k]["train"]
    n_conditions = 1 if k in (1, 3) else 6

    d, top_sensors, correlations, pca_variance = prepare_train(
        train, n_conditions
    )

    hi_rul = d["HI"].corr(d["RUL"], method="spearman")

    prepared[k] = {
        "data": d,
        "top_sensors": top_sensors,
        "correlations": correlations,
        "pca_variance": pca_variance,
        "hi_rul": hi_rul
    }

    hi_results.append({
        "dataset": f"FD00{k}",
        "Spearman_HI_RUL": hi_rul,
        "PC1_explained_variance": pca_variance,
        "top_sensors": ", ".join(top_sensors)
    })

hi_results_df = pd.DataFrame(hi_results)
hi_results_df

## 6. Visualização do Health Index

A correlação positiva entre HI e RUL indica que, em geral, o índice diminui conforme a unidade se aproxima do final da trajetória de degradação.


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(
    hi_results_df["dataset"],
    hi_results_df["Spearman_HI_RUL"]
)
plt.ylim(0, 1)
plt.ylabel("Correlação de Spearman: HI × RUL")
plt.title("Validação do Health Index")
plt.grid(axis="y", alpha=0.2)
plt.show()

### Exemplo: trajetória de uma unidade do FD001


In [ ]:
fd1 = prepared[1]["data"]
example_unit = fd1["unit"].iloc[0]
g = fd1[fd1["unit"] == example_unit]

plt.figure(figsize=(10, 4))
plt.plot(g["cycle"], g["HI"], label="Health Index")
plt.xlabel("Ciclo")
plt.ylabel("HI")
plt.ylim(0, 1)
plt.title(f"FD001 — evolução do Health Index — unidade {example_unit}")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

## 7. Estados do MDP

O Health Index é discretizado em:

| Estado | Condição | Faixa |
|---|---|---|
| S0 | Saudável | HI > 0,75 |
| S1 | Moderado | 0,50 < HI ≤ 0,75 |
| S2 | Severo | 0,25 < HI ≤ 0,50 |
| S3 | Crítico | HI ≤ 0,25 |
| S4 | Falha | final da trajetória |

As probabilidades naturais de transição são estimadas diretamente das sequências de estados observadas.

Como a degradação latente simulada é progressiva, pequenas reversões produzidas pelo Health Index são tratadas como ruído: sua massa de probabilidade é incorporada à permanência no estado atual.


In [ ]:
def transition_matrix(d):
    counts = np.zeros((5, 5), dtype=int)

    for _, g in d.groupby("unit"):
        states = g.sort_values("cycle")["state"].to_numpy()

        for current_state, next_state in zip(states[:-1], states[1:]):
            counts[current_state, next_state] += 1

        # O final de cada trajetória de treino é conectado à falha.
        counts[states[-1], 4] += 1

    P = np.zeros((5, 5), dtype=float)

    for state in range(4):
        if counts[state].sum():
            P[state] = counts[state] / counts[state].sum()

    P[4, 4] = 1.0

    # Limpeza monotônica das reversões do HI.
    for state in range(4):
        P[state, state] += P[state, :state].sum()
        P[state, :state] = 0.0

    return counts, P

In [ ]:
transition_results = {}

for k in range(1, 5):
    counts, Pnatural = transition_matrix(prepared[k]["data"])
    transition_results[k] = (counts, Pnatural)

print("Matriz de transição natural — FD001")
display(pd.DataFrame(
    transition_results[1][1],
    index=STATE_NAMES,
    columns=STATE_NAMES
).round(4))

## 8. Ambiente de decisão

Ações:

- **operate**: continuar operando;
- **reduced_load**: operar com carga reduzida;
- **maintenance**: manutenção preventiva.

Cenário-base:

- operar: **+10**
- carga reduzida: **+6**
- manutenção preventiva: **−25**
- falha/reparo corretivo: **−100**
- fator de degradação sob carga reduzida: **0,5**
- fator de desconto: **γ = 0,95**

Esses valores são **unidades de utilidade experimentais**, não custos reais e não fazem parte do dataset da NASA.


In [ ]:
def build_mdp(
    Pnatural,
    degrade_factor=.5,
    r_oper=10,
    r_reduced=6,
    c_maint=25,
    c_fail=100
):
    P = np.zeros((5, 3, 5))
    R = np.zeros((5, 3, 5))

    # Estado de falha: reparo corretivo e retorno ao saudável.
    for action in range(3):
        P[4, action, 0] = 1
        R[4, action, 0] = -c_fail

    for state in range(4):
        # Operar normalmente.
        P[state, 0] = Pnatural[state]
        R[state, 0, :] = r_oper
        R[state, 0, 4] = r_oper - c_fail

        # Carga reduzida: reduz a probabilidade de piora.
        row = Pnatural[state].copy()
        worse_states = list(range(state + 1, 5))
        moved_probability = row[worse_states].sum() * (1 - degrade_factor)
        row[worse_states] *= degrade_factor
        row[state] += moved_probability

        P[state, 1] = row
        R[state, 1, :] = r_reduced
        R[state, 1, 4] = r_reduced - c_fail

        # Manutenção preventiva: retorno ao saudável.
        P[state, 2, 0] = 1
        R[state, 2, 0] = -c_maint

    return P, R

## 9. Value Iteration

A atualização utilizada é:

\[
V_{k+1}(s)=\max_a \sum_{s'}P(s'|s,a)
[R(s,a,s')+\gamma V_k(s')]
\]


In [ ]:
def value_iteration(P, R, gamma=.95, tol=1e-10):
    V = np.zeros(5)

    for iteration in range(10000):
        Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
        new_V = Q.max(axis=1)

        if np.max(np.abs(new_V - V)) < tol:
            V = new_V
            break

        V = new_V

    Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
    policy = Q.argmax(axis=1)

    return V, Q, policy

## 10. Q-Learning

O Q-Learning utiliza:

\[
Q(s,a) \leftarrow Q(s,a) +
\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]
\]

Configuração:

- 30.000 episódios;
- 60 passos por episódio;
- exploração ε-greedy;
- ε inicial = 1;
- ε mínimo = 0,02;
- semente = 42.


In [ ]:
def sample_next(probabilities, rng):
    x = rng.random()
    accumulated = 0.0

    for i, probability in enumerate(probabilities):
        accumulated += probability
        if x <= accumulated:
            return i

    return len(probabilities) - 1


def q_learning(P, R, gamma=.95, episodes=30000, steps=60, seed=42):
    rng = random.Random(seed)

    Q = [[0.0] * 3 for _ in range(5)]
    visits = [[0] * 3 for _ in range(5)]

    epsilon = 1.0

    for _ in range(episodes):
        # Exploring starts.
        state = rng.randrange(5)

        for _ in range(steps):
            if rng.random() < epsilon:
                action = rng.randrange(3)
            else:
                action = max(range(3), key=lambda a: Q[state][a])

            next_state = sample_next(P[state, action], rng)
            reward = float(R[state, action, next_state])

            visits[state][action] += 1
            alpha = 1 / (visits[state][action] ** 0.6)

            Q[state][action] += alpha * (
                reward
                + gamma * max(Q[next_state])
                - Q[state][action]
            )

            state = next_state

        epsilon = max(.02, epsilon * .9997)

    Q = np.asarray(Q)
    policy = Q.argmax(axis=1)

    return Q, policy

## 11. Avaliação das políticas


In [ ]:
def policy_value(P, R, policy, gamma=.95):
    Ppi = np.array([P[s, policy[s]] for s in range(5)])
    rpi = np.array([
        (P[s, policy[s]] * R[s, policy[s]]).sum()
        for s in range(5)
    ])

    return np.linalg.solve(
        np.eye(5) - gamma * Ppi,
        rpi
    )

In [ ]:
experiment_results = []
policies = {}

baseline = np.array([0, 0, 2, 2, 0])

for k in range(1, 5):
    Pnatural = transition_results[k][1]
    P, R = build_mdp(Pnatural)

    V, Q_vi, policy_vi = value_iteration(P, R)
    Q_ql, policy_ql = q_learning(P, R)

    baseline_return = policy_value(P, R, baseline)[0]
    vi_return = policy_value(P, R, policy_vi)[0]
    ql_return = policy_value(P, R, policy_ql)[0]

    policies[k] = {
        "P": P,
        "R": R,
        "VI": policy_vi,
        "QL": policy_ql
    }

    experiment_results.append({
        "dataset": f"FD00{k}",
        "baseline_return": baseline_return,
        "VI_return": vi_return,
        "QL_return": ql_return,
        "VI_policy": " → ".join(ACTIONS[a] for a in policy_vi[:4]),
        "QL_policy": " → ".join(ACTIONS[a] for a in policy_ql[:4])
    })

results_df = pd.DataFrame(experiment_results)
results_df

## 12. Política encontrada

No cenário-base, Value Iteration e Q-Learning convergem para a mesma política operacional:

- saudável → **operar**
- moderado → **operar**
- severo → **operar**
- crítico → **manutenção**

A baseline é mais conservadora:

- saudável → operar
- moderado → operar
- severo → manutenção
- crítico → manutenção

A ação de carga reduzida não é selecionada pela política ótima neste cenário.


In [ ]:
policy_table = pd.DataFrame({
    "Estado": STATE_NAMES[:4],
    "Baseline": [ACTIONS[a] for a in baseline[:4]],
    "Value Iteration": [ACTIONS[a] for a in policies[1]["VI"][:4]],
    "Q-Learning": [ACTIONS[a] for a in policies[1]["QL"][:4]]
})
policy_table

## 13. Comparação do retorno descontado


In [ ]:
plot_df = results_df.set_index("dataset")[
    ["baseline_return", "VI_return", "QL_return"]
]

ax = plot_df.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Retorno descontado a partir do estado saudável")
ax.set_xlabel("Dataset")
ax.set_title("Comparação das políticas")
ax.grid(axis="y", alpha=0.2)
plt.xticks(rotation=0)
plt.show()

results_df[[
    "dataset", "baseline_return", "VI_return", "QL_return"
]]

## 14. Simulação de longo prazo

Além do retorno descontado, avaliamos frequência de manutenção, falhas, carga reduzida e disponibilidade. Isso evidencia o trade-off entre maior utilização e maior risco.


In [ ]:
def simulate_policy(P, R, policy, steps=50000, seed=2026):
    rng = random.Random(seed)
    state = 0

    reward_sum = 0.0
    failures = 0
    maintenance = 0
    reduced_load = 0
    operational_steps = 0

    for _ in range(steps):
        action = int(policy[state])

        if state != 4:
            operational_steps += 1
            if action == 2:
                maintenance += 1
            elif action == 1:
                reduced_load += 1

        next_state = sample_next(P[state, action], rng)
        reward_sum += float(R[state, action, next_state])

        if next_state == 4:
            failures += 1

        state = next_state

    return {
        "AvgReward": reward_sum / steps,
        "Failures_per_10k": failures / steps * 10000,
        "Maintenance_per_10k": maintenance / steps * 10000,
        "ReducedLoad_pct": reduced_load / steps * 100,
        "Availability": operational_steps / steps
    }

In [ ]:
long_run = []

for k in range(1, 5):
    P = policies[k]["P"]
    R = policies[k]["R"]

    for name, policy in [
        ("Baseline", baseline),
        ("Value Iteration", policies[k]["VI"]),
        ("Q-Learning", policies[k]["QL"])
    ]:
        metrics = simulate_policy(P, R, policy)
        long_run.append({
            "Dataset": f"FD00{k}",
            "Policy": name,
            **metrics
        })

long_run_df = pd.DataFrame(long_run)
long_run_df

### Exemplo do trade-off no FD004

No cenário-base, a política otimizada posterga a manutenção. Isso reduz o número de intervenções, mas pode aumentar a exposição a falhas.

Esse resultado é importante porque mostra que **maior retorno descontado não significa automaticamente maior segurança**. A política depende da função de recompensa escolhida.


In [ ]:
long_run_df[long_run_df["Dataset"] == "FD004"]

## 15. Exportação dos resultados


In [ ]:
results_df.to_csv("resultados_notebook.csv", index=False)
hi_results_df.to_csv("health_index_resultados.csv", index=False)
long_run_df.to_csv("simulacao_longo_prazo.csv", index=False)

for k in range(1, 5):
    pd.DataFrame(
        transition_results[k][1],
        index=STATE_NAMES,
        columns=STATE_NAMES
    ).to_csv(f"transitions_FD00{k}.csv")

print("Arquivos de resultados exportados.")

## 16. Conclusão

O experimento demonstrou um pipeline completo para transformar séries temporais de degradação em decisões de manutenção.

1. Os sensores do C-MAPSS foram pré-processados e normalizados.
2. Um Health Index foi construído por seleção de sensores e PCA.
3. O Health Index foi convertido em estados discretos.
4. As trajetórias foram utilizadas para estimar probabilidades naturais de degradação.
5. Foi construído um MDP de manutenção.
6. Value Iteration e Q-Learning convergiram para a mesma política no cenário-base.
7. A política otimizada apresentou maior retorno descontado que a baseline, mas os cenários mais complexos evidenciaram um trade-off entre redução de manutenção e exposição a falhas.

### Limitações

- C-MAPSS é um conjunto simulado de turbofans.
- Não representa fisicamente uma linha de produção.
- O dataset não contém decisões históricas de manutenção.
- Custos, recompensas e efeitos das ações são parâmetros do modelo.
- Uma aplicação industrial real exigiria recalibração com dados e restrições da empresa.

**Ideia central:** mais do que prever quando um equipamento irá falhar, o objetivo é usar a informação de degradação para decidir o que fazer antes da falha.


# PARTE II — Research V2: Robustez e Validação Experimental

Esta parte amplia o experimento original sem substituir seus resultados. O objetivo é verificar quando as conclusões permanecem válidas diante de mudanças de amostra, discretização, função de recompensa, representação do Health Index e aleatoriedade do Q-Learning.

A V2 investiga: validação fora da amostra por motores; bootstrap da matriz de transição; sensibilidade de recompensas; ablação do pipeline; sensibilidade dos thresholds; Health Index alternativo; múltiplas seeds do Q-Learning; baselines adicionais; cenários de estresse; e a hipótese de transição monotônica.

> O C-MAPSS continua sendo usado como fonte de trajetórias simuladas de degradação. Estes experimentos não constituem validação industrial externa.


## 17. Funções auxiliares de robustez


In [ ]:
from sklearn.model_selection import KFold

def policy_signature(policy):
    return " → ".join(ACTIONS[int(a)] for a in policy[:4])

def transition_matrix_v2(d, monotonic=True):
    counts=np.zeros((5,5),dtype=int)
    for _,g in d.groupby("unit"):
        st=g.sort_values("cycle")["state"].to_numpy()
        for a,b in zip(st[:-1],st[1:]): counts[a,b]+=1
        counts[st[-1],4]+=1
    P=np.zeros((5,5),float)
    for s in range(4):
        if counts[s].sum(): P[s]=counts[s]/counts[s].sum()
    P[4,4]=1
    if monotonic:
        for s in range(4):
            P[s,s]+=P[s,:s].sum(); P[s,:s]=0
    return counts,P

def assign_states(d, thresholds=(.25,.50,.75), hi_col="HI"):
    a,b,c=thresholds
    out=d.copy()
    out["state"]=pd.cut(out[hi_col],[-np.inf,a,b,c,np.inf],labels=[3,2,1,0]).astype(int)
    return out

def exact_policy_metrics(P,R,policy,gamma=.95):
    Ppi=np.array([P[s,int(policy[s])] for s in range(5)])
    rpi=np.array([(P[s,int(policy[s])]*R[s,int(policy[s])]).sum() for s in range(5)])
    discounted=np.linalg.solve(np.eye(5)-gamma*Ppi,rpi)[0]
    mu=np.zeros(5); mu[0]=1
    for _ in range(200000):
        new=mu@Ppi
        if np.max(np.abs(new-mu))<1e-14: mu=new; break
        mu=new
    fail=sum(mu[s]*Ppi[s,4] for s in range(5))
    maint=sum(mu[s] for s in range(4) if int(policy[s])==2)
    reduced=sum(mu[s] for s in range(4) if int(policy[s])==1)
    return {"discounted_return":discounted,"avg_reward":float(mu@rpi),
            "failures_per_10k":fail*10000,"maintenance_per_10k":maint*10000,
            "reduced_load_pct":reduced*100,"availability":1-mu[4]}


## 18. Validação 5-fold por motores

A divisão é feita por unidade, e não por linhas. Na execução V2 validada, a correlação média HI×RUL nos motores retidos foi **0,776 / 0,783 / 0,841 / 0,778** para FD001–FD004. O retorno médio da política derivada do treino superou a baseline nos quatro subconjuntos.

Os resultados detalhados estão em `results/research_v2/heldout_5fold_validation.csv`.


In [ ]:
heldout_validated = pd.DataFrame({
    "dataset":["FD001","FD002","FD003","FD004"],
    "heldout_HI_RUL":[0.776182,0.783353,0.840679,0.777850],
    "VI_return":[198.601304,197.536813,198.293576,196.806882],
    "baseline_return":[196.012978,193.520096,197.155896,196.077545]
})
heldout_validated["gain"]=heldout_validated["VI_return"]-heldout_validated["baseline_return"]
heldout_validated


## 19. Bootstrap da incerteza das transições

Foram realizadas **1.000 reamostragens de motores por subconjunto (4.000 no total)**. A construção do HI foi mantida fixa para isolar a incerteza amostral de P(s'|s). A política-base de Value Iteration permaneceu igual em **100% das 4.000 reamostragens**.


In [ ]:
bootstrap_validated = pd.DataFrame({
    "dataset":["FD001","FD002","FD003","FD004"],
    "policy_stability":[1.0,1.0,1.0,1.0],
    "return_mean":[198.569,197.520,198.249,197.098],
    "return_sd":[0.111,0.096,0.278,0.287],
    "p_critical_failure_mean":[0.0520,0.0566,0.0503,0.0489]
})
bootstrap_validated


## 20. Sensibilidade da função de recompensa

A grade V2 avaliou **225 configurações por subconjunto, 900 no total**, variando custo de manutenção, custo de falha, recompensa de carga reduzida e efetividade da carga reduzida.

A política-base apareceu em aproximadamente **95,6% / 95,6% / 79,6% / 67,1%** das configurações de FD001–FD004. FD003/FD004 são mais sensíveis ao custo relativo da falha. A carga reduzida foi ótima apenas em uma pequena região do espaço testado.


In [ ]:
sensitivity_validated=pd.DataFrame({
 "dataset":["FD001","FD002","FD003","FD004"],
 "configs":[225]*4,
 "base_policy_count":[215,215,179,151],
 "earlier_maintenance_count":[0,0,36,63],
 "always_operate_count":[8,8,8,8],
 "other_or_reduced_load_count":[2,2,2,3]
})
sensitivity_validated["base_policy_rate"]=sensitivity_validated["base_policy_count"]/225
sensitivity_validated


## 21. Ablation study

A V2 comparou o pipeline completo com: ausência de normalização por regime; todos os sensores não constantes; e ausência de suavização temporal.

O resultado mais forte aparece em FD002/FD004: sem normalização por regime, Spearman HI×RUL cai de **0,782 para 0,045** e de **0,774 para 0,085**, respectivamente. Isso fornece justificativa empírica para a normalização por condição operacional.


In [ ]:
ablation_key=pd.DataFrame({
 "dataset":["FD002","FD004"],
 "full_HI_RUL":[0.7825,0.7742],
 "without_regime_normalization_HI_RUL":[0.045,0.085]
})
ablation_key


## 22. Sensibilidade dos thresholds

Foram testadas cinco discretizações. A política-base permaneceu igual em **19 de 20** combinações dataset/threshold. A exceção ocorreu no FD003 com thresholds (0,20; 0,40; 0,60), quando a manutenção foi antecipada.

A discretização é, portanto, relativamente robusta, mas continua sendo uma hipótese metodológica relevante.


## 23. Health Index alternativo

PCA foi comparado com um índice simples construído pela média de sensores selecionados após orientação pelo sinal da associação com RUL. Os dois métodos produziram **a mesma política de Value Iteration nos quatro subconjuntos**.

Isso indica que a conclusão decisória principal não depende exclusivamente do PCA.


## 24. Estabilidade do Q-Learning por seed

A análise de robustez repetiu Q-Learning com **20 seeds por subconjunto**, usando 2.000 episódios por seed para testar sensibilidade de convergência.

A coincidência com Value Iteration foi: **100% FD001, 100% FD002, 85% FD003 e 75% FD004**. Logo, a afirmação original deve ser refinada: FD003/FD004 são mais sensíveis à seed e ao orçamento de treinamento.


In [ ]:
qlearning_stability=pd.DataFrame({
 "dataset":["FD001","FD002","FD003","FD004"],
 "seeds":[20]*4,"matches_VI":[20,20,17,15]
})
qlearning_stability["match_rate"]=qlearning_stability["matches_VI"]/qlearning_stability["seeds"]
qlearning_stability


## 25. Baselines adicionais e cenários de estresse

Além da baseline original, a V2 considera manutenção tardia e uma heurística de carga reduzida. Sob a utilidade-base, Value Iteration coincide com a heurística de manutenção tardia.

Nos cenários de estresse, custo alto de falha antecipa manutenção em FD003/FD004; manutenção barata também antecipa a intervenção em FD004. Isso mostra por que a política-base não deve ser tratada como regra universal.


## 26. Hipótese de transição monotônica

A versão original redireciona pequenas reversões do HI para a permanência no estado atual. A V2 comparou essa matriz com as transições observadas sem correção. A política ótima permaneceu igual nos quatro subconjuntos, embora os retornos mudem ligeiramente.

Portanto, a política-base não é um artefato dessa hipótese de denoising.


## 27. Resumo quantitativo da robustez


In [ ]:
robustness_summary=pd.DataFrame({
 "dataset":["FD001","FD002","FD003","FD004"],
 "heldout_HI_RUL":[0.776182,0.783353,0.840679,0.777850],
 "heldout_return_gain":[2.588326,4.016717,1.137680,0.729337],
 "bootstrap_policy_stability":[1.0,1.0,1.0,1.0],
 "bootstrap_return_sd":[0.1107,0.0956,0.2779,0.2869],
 "qlearning_VI_match_rate":[1.0,1.0,0.85,0.75],
 "reward_grid_base_policy_rate":[0.9556,0.9556,0.7956,0.6711]
})
robustness_summary


# 28. Conclusão da Research V2

A versão robusta permite uma conclusão mais cuidadosa:

> **Sob a construção de Health Index e as hipóteses do ambiente de manutenção adotadas, operar até o estado severo e realizar manutenção preventiva no estado crítico é uma política estável à reamostragem dos motores, à maioria das discretizações testadas, a uma representação alternativa do Health Index e a uma região ampla dos parâmetros de utilidade. As implicações de confiabilidade, entretanto, dependem fortemente do custo relativo atribuído à falha, especialmente em FD003 e FD004.**

### Evidências favoráveis
- validação fora da amostra por motores manteve forte associação HI×RUL;
- 4.000 bootstraps mantiveram a política de VI;
- 19/20 discretizações mantiveram a política-base;
- um HI alternativo produziu a mesma política;
- a normalização por regime foi empiricamente justificada em FD002/FD004.

### Limitações reveladas
- a política depende da função de recompensa;
- FD003/FD004 apresentam maior sensibilidade do Q-Learning;
- maior retorno descontado pode coexistir com maior exposição a falhas;
- C-MAPSS é simulado e não contém decisões históricas de manutenção.

A conclusão é válida para **o ambiente experimental desta pesquisa**, não como recomendação direta para uma instalação industrial real.
